# Intel Lab Dataset Demo — Interactive

Run each cell as you explain. Ask students questions after each visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, urllib.request, gzip

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100

## Understanding the Dataset

**What's in the Intel Lab dataset?**

54 sensors deployed across a UC Berkeley computer science lab for 38 days (2004).

Each sensor measures:
- **temperature** (°C)
- **humidity** (%)
- **light** (lux)
- **voltage** (battery level)

Total: 2.3 million readings

**How to explore it with pandas:**

## Load the data

In [ ]:
# Explore the dataset with pandas
print('Dataset shape:', df.shape)
print('\nFirst 5 rows:')
print(df.head())
print('\nData types:')
print(df.dtypes)
print('\nBasic statistics:')
print(df[['temperature', 'humidity', 'light', 'voltage']].describe())

In [ ]:
# Download if needed
data_path = 'data.txt'
if not os.path.exists(data_path):
    print('Downloading Intel Lab dataset...')
    url = 'https://db.csail.mit.edu/labdata/data.txt.gz'
    urllib.request.urlretrieve(url, 'data.txt.gz')
    with gzip.open('data.txt.gz', 'rb') as f_in:
        with open(data_path, 'wb') as f_out:
            f_out.write(f_in.read())
    print('Done.')

# Load
columns = ['date', 'time', 'epoch', 'moteid', 'temperature', 'humidity', 'light', 'voltage']
df = pd.read_csv(data_path, sep=r'\s+', names=columns, na_values=[''])
df['timestamp'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')
df = df.dropna(subset=['timestamp'])
df = df[df['moteid'].between(1, 54)]

print(f'Loaded {len(df):,} rows')
print(f'Date range: {df["timestamp"].min()} to {df["timestamp"].max()}')
print(f'Sensors: {df["moteid"].nunique()}')

---
## EXERCISE 1: Data Quality Audit

### Question 1: How many readings per sensor?

In [ ]:
**Ask:** Why does Mote 5 have only 35 readings while Mote 1 has 8,000+? What could have happened?

In [ ]:
# Histogram
fig, ax = plt.subplots()
readings_per_mote.hist(bins=30, ax=ax)
ax.set_xlabel('Number of readings')
ax.set_ylabel('Number of sensors')
ax.set_title('How many readings does each sensor have?')
plt.tight_layout()
plt.show()

print(f'\nMote 5 has {readings_per_mote[5]} readings')
print(f'Mote 1 has {readings_per_mote[1]} readings')

**QUESTION FOR STUDENTS:** Why does Mote 5 have only 35 readings while Mote 1 has 8,000+? What could have happened?

**Ask:** The lab is maintained at 20-22°C. We have 18% of readings above 40°C. Are these real temperatures or sensor errors? Why?

In [ ]:
# Full distribution
fig, ax = plt.subplots()
df['temperature'].hist(bins=100, ax=ax)
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Count')
ax.set_title('Temperature distribution — full range')
ax.axvline(x=40, color='red', linestyle='--', linewidth=2, label='40°C threshold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Count above 40°C
above_40 = (df['temperature'] > 40).sum()
total = df['temperature'].notna().sum()
pct = 100 * above_40 / total

print(f'Total readings: {total:,}')
print(f'Above 40°C: {above_40:,} ({pct:.1f}%)')
print(f'\nLab temperature: 20-22°C')
print(f'Readings above 40°C: {pct:.1f}%')

**Ask:** When did Mote 49 stop? Was it a gradual failure or sudden? What does the pattern tell you?

### Question 3: Do sensors stop reporting?

In [ ]:
# Time series for Mote 49
mote49 = df[df['moteid'] == 49].sort_values('timestamp')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(mote49['timestamp'], mote49['temperature'], 'b-', alpha=0.7, markersize=2)
ax.set_xlabel('Timestamp')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Mote 49 — does it stop reporting?')
ax.set_ylim(15, 50)
plt.tight_layout()
plt.show()

last_reading = mote49['timestamp'].max()
print(f'Mote 49 last reading: {last_reading}')
print(f'Total deployment: 38 days')
print(f'Mote 49 died after: {(last_reading - mote49["timestamp"].min()).days} days')

**QUESTION FOR STUDENTS:** When did Mote 49 stop? Was it a gradual failure or sudden? What does the pattern tell you?

---
## EXERCISE 2: Compare Sensors

### Question 1: Do nearby sensors agree?

In [ ]:
**Ask:** Look at the heatmap. Motes 1 and 2 have correlation 0.98+. Mote 34 is lower. Why?

**QUESTION FOR STUDENTS:** Look at the heatmap. Motes 1 and 2 have correlation 0.98+. Mote 34 is lower. Why?

### Question 2: What's the spread between sensors?

In [ ]:
**Ask:** The max spread is 120°C but the median is <1°C. Why such a huge difference? Which number matters?

**QUESTION FOR STUDENTS:** The max spread is 120°C but the median is <1°C. Why such a huge difference? Which number matters?

---
## EXERCISE 3: Anomaly Detection

### Two methods on the same data

In [ ]:
**Ask:** Fixed threshold finds 6,700. Z-score finds 500. Only 10 overlap. Why such a huge difference?

In [ ]:
# Visualization
fig, ax = plt.subplots(figsize=(14, 5))

# All readings
ax.plot(mote1.index, mote1['temperature'], 'b-', alpha=0.3, label='Temperature', linewidth=0.5)

# Fixed threshold anomalies
fixed_anomalies = mote1[mote1['anomaly_fixed']]
ax.scatter(fixed_anomalies.index, fixed_anomalies['temperature'], 
           c='red', s=10, alpha=0.5, label=f'Fixed threshold ({fixed_count:,})')

# Z-score only
zscore_only = mote1[mote1['anomaly_zscore'] & ~mote1['anomaly_fixed']]
ax.scatter(zscore_only.index, zscore_only['temperature'],
           c='green', s=20, alpha=0.7, label=f'Z-score only ({len(zscore_only):,})', marker='^')

# Thresholds
ax.axhline(y=15, color='orange', linestyle='--', alpha=0.5, linewidth=1.5)
ax.axhline(y=35, color='orange', linestyle='--', alpha=0.5, linewidth=1.5, label='Fixed thresholds')

ax.set_xlabel('Timestamp')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Mote 1: Fixed threshold vs Z-score')
ax.legend(loc='upper right')
ax.set_ylim(10, 50)

plt.tight_layout()
plt.show()

**QUESTION FOR STUDENTS:** 
- Red dots: caught by fixed threshold. Green triangles: caught by z-score only.
- Why are the numbers so different (6,700 vs 500)?
- Which method would you use for safety alerts? Which for drift detection?